## 🔮 TFB Prediction Tutorial 4/4: Model Inference - From Trained Model to Predictions

Welcome to the final tutorial in our series on fine-tuning a Genomic Foundation Model. We have come a long way:

1.  We prepared the DeepSEA dataset (`01_data_preparation.ipynb`).
2.  We initialized a model architecture suitable for our task (`02_model_initialization.ipynb`).
3.  We trained the model and saved the best-performing checkpoint (`03_model_training.ipynb`).

Now, we have a powerful, fine-tuned model stored as `best_model.pth`. But a model is only useful if we can use it to make predictions on new, unseen data. This process is called **inference**.

In this tutorial, we will cover:
1.  **The Inference Pipeline**: Understanding the essential steps for getting predictions.
2.  **Inference with `ModelHub`**: The easy, one-line way to load and predict with `OmniGenBench`.
3.  **Manual Inference**: A more hands-on approach for custom data processing workflows.
4.  **Final Evaluation**: Assessing our model's performance on the held-out test set.
5.  **Deployment Concepts**: A brief look at how to serve your model as an API.

By the end, you will be able to use your trained model to predict transcription factor binding sites and understand how to integrate it into larger applications.

### 1. The Inference Pipeline

Making a prediction with a trained model involves a clear, logical sequence of steps. It's crucial that the data processing during inference **exactly matches** the processing used during training.

The pipeline looks like this:

1.  **Load Model and Tokenizer**: You must load the exact model checkpoint (`best_model.pth`) that you saved during training. You also need the same tokenizer that was used to prepare the training data.
2.  **Prepare Input**: Take a new, raw DNA sequence.
3.  **Tokenize**: Use the loaded tokenizer to convert the DNA sequence into `input_ids` and an `attention_mask`, just as we did for the training data. This includes applying the same `max_length`, padding, and truncation strategies.
4.  **Predict**: Pass the tokenized input through the model to get the raw output scores (logits).
5.  **Post-process**: Convert the logits into a more interpretable format, such as probabilities (by applying a Sigmoid function) or binary predictions (by applying a threshold like 0.5).

**`OmniGenBench` simplify all the processes and provides a unified interface for inference.**

### 2. Inference with `ModelHub`: The Easy Way

For many standard use cases, `OmniGenBench` offers a high-level `ModelHub` API that encapsulates the entire inference pipeline. It allows you to load a fine-tuned model and get predictions with a single line of code.

`ModelHub` automatically handles loading the correct model architecture, the tokenizer, and the saved weights from your checkpoint directory.

Let's see it in action.

In [ ]:
from omnigenbench import ModelHub

# Define a sample sequence to test
sequence = "GATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAGATTACAG..."

# Model path or indentifier on huggingface
model_to_load = 'ogb_tfb_finetuned'

model = ModelHub.load(model_to_load)

# Make predictions
logits = model.inference(sequence)

print("Inference successful!")
print(f"Output logits shape: {logits.shape}")

This approach is incredibly convenient for standard models and tasks. You simply point `ModelHub` to your checkpoint directory, and it takes care of the rest.

### 3. Manual Inference: For Custom Workflows

While `ModelHub` is convenient, sometimes you need more control. For example, you might have a custom data loading pipeline or want to perform inference on a large dataset more efficiently. In these cases, you can perform the inference steps manually.

This involves loading the model and tokenizer yourself and then explicitly tokenizing the input before passing it to the model. This is essentially what `ModelHub` does under the hood.

#### 3.1. Setup: Re-initializing Model, Tokenizer, and Data

First, let's set up our environment again. We need to load the necessary libraries and define our configuration. We also need our test dataloader to evaluate the model on the full test set.

In [ ]:
# Import libraries
import torch
import numpy as np
import os
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

from omnigenbench import (
    OmniTokenizer,
    OmniModelForMultiLabelSequenceClassification,
    OmniDatasetForSequenceClassification
)
from autocuda import auto_cuda

# Inference Configuration
class TFBInferenceConfig:
    MODEL_NAME = "yangheng/OmniGenome-52M"
    DATASET_NAME = "yangheng/deepsea_tfb_prediction"
    CHECKPOINT_DIR = "./checkpoints/deepsea_finetune"
    
    NUM_LABELS = 919
    MAX_LENGTH = 512
    BATCH_SIZE = 64
    DEVICE = auto_cuda()

config = TFBInferenceConfig()

print("✅ Libraries imported and configuration set!")
print(f"🎯 Device: {config.DEVICE}")
print(f"🧬 Model: {config.MODEL_NAME}")
print(f"📊 Labels: {config.NUM_LABELS}")

In [ ]:
# Load tokenizer and test dataset
print("🔄 Loading tokenizer...")
tokenizer = OmniTokenizer.from_pretrained(config.MODEL_NAME, trust_remote_code=True)

print("📊 Loading test dataset...")
datasets = OmniDatasetForSequenceClassification.from_huggingface(
    dataset_name=config.DATASET_NAME,
    tokenizer=tokenizer,
    max_length=config.MAX_LENGTH,
    cache_dir="deepsea_tfb_cache"
)

test_loader = datasets['test'].get_dataloader(
    batch_size=config.BATCH_SIZE, 
    shuffle=False
)

print(f"🧪 Test dataset: {len(datasets['test'])} samples")

# Load model (try trained checkpoint first, fallback to base model)
print("🔄 Loading model...")
try:
    # Try to load the latest checkpoint
    checkpoint_files = [f for f in os.listdir(config.CHECKPOINT_DIR) if f.endswith('.pth')]
    if checkpoint_files:
        latest_checkpoint = max(checkpoint_files)
        checkpoint_path = os.path.join(config.CHECKPOINT_DIR, latest_checkpoint)
        
        model = OmniModelForMultiLabelSequenceClassification.from_pretrained(
            config.MODEL_NAME,
            num_labels=config.NUM_LABELS,
            trust_remote_code=True
        )
        model.load_state_dict(torch.load(checkpoint_path, map_location=config.DEVICE))
        print(f"✅ Loaded trained model from: {checkpoint_path}")
    else:
        model = OmniModelForMultiLabelSequenceClassification.from_pretrained(
            config.MODEL_NAME,
            num_labels=config.NUM_LABELS,
            trust_remote_code=True
        )
        print(f"⚠️  No checkpoint found, using base model")
except FileNotFoundError:
    model = OmniModelForMultiLabelSequenceClassification.from_pretrained(
        config.MODEL_NAME,
        num_labels=config.NUM_LABELS,
        trust_remote_code=True
    )
    print(f"⚠️  Checkpoint directory not found, using base model")

model.to(config.DEVICE)
model.eval()
print("✅ Model loaded and ready for inference!")

#### 3.2. Loading the Fine-Tuned Model

Now, we load the model. We first initialize the model architecture using `OmniModelForMultiLabelSequenceClassification` and then load our fine-tuned weights from the `best_model.pth` file.

In [ ]:
# Initialize the model architecture
model = ModelHub.load(model_to_load)

print("Fine-tuned model loaded successfully.")

#### 3.3. The Inference Loop

With the model and dataloader ready, we can now loop through the test set, make predictions, and store the results. This is the core of manual inference.

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():  # Disable gradient calculations for efficiency
    for batch in tqdm(test_dataloader, desc="Running Inference"):
        inputs, labels = batch
        inputs = {k: v.to(config["device"]) for k, v in inputs.items()}
        
        outputs = model.inference(**inputs)
        logits = outputs['logits']
        
        all_preds.append(logits.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

print("Inference on the test set complete.")

This final score gives you a reliable estimate of how well your model will perform in a real-world scenario on new genomic data.

### 4. Deployment Concepts: Serving Your Model

A trained model in a notebook is great for research, but for real-world applications, you'll want to **deploy** it as a service. This typically means wrapping it in a web API. A popular choice for this is **FastAPI**.

The concept is simple:
1.  **Create a FastAPI App**: A simple Python script that defines API endpoints.
2.  **Load the Model**: In the app's startup logic, you would load your fine-tuned model and tokenizer once (e.g., using the `ModelHub` or the manual method).
3.  **Define a Prediction Endpoint**: Create an endpoint (e.g., `/predict`) that accepts a DNA sequence as input.
4.  **Process and Predict**: Inside the endpoint function, you would call the model's inference method with the input sequence and return the prediction as a JSON response.

While a full deployment tutorial is beyond our current scope, the `ModelHub` API is designed to make this transition as smooth as possible.

### Summary and Conclusion

This tutorial marks the end of our journey from a biological question to a fully trained and evaluated model. We have covered the complete lifecycle: data preparation, model initialization, training, and now, inference.

You have learned how to:
-   Understand the end-to-end inference pipeline.
-   Use the high-level `ModelHub` for quick and easy predictions.
-   Perform manual inference for greater control and batch processing.
-   Evaluate the final model on a test set to get a definitive performance metric.
-   Conceptualize how to deploy your model as a service.

You are now equipped with the fundamental skills to tackle your own genomic prediction tasks using `OmniGenBench`. You can return to the main tutorial or explore the other examples in the repository to learn about different tasks and models. Happy modeling!